# Finance Case Study
Companion notebook for the **Finance Case Study** section of Chapter 11.

Pattern: **researcher, drafter, reviewer**. Agents prepare, compare, and flag; humans approve, decide, and act -- final decisions on payments, filings, approvals, or account actions never happen without a human sign-off.

This is the book's `approve_finance_output`, unchanged:

In [1]:
def approve_finance_output(task):
    if task["impact_level"] == "high":
        return "human_review_required"
    if not task["all_numbers_verified"]:
        return "reject"
    if not task["policy_check_passed"]:
        return "reject"
    return "send_to_manager"

In [2]:
cases = [
    {"name": "routine expense summary", "impact_level": "low", "all_numbers_verified": True, "policy_check_passed": True},
    {"name": "credit limit change recommendation", "impact_level": "high", "all_numbers_verified": True, "policy_check_passed": True},
    {"name": "draft memo with an unverified figure", "impact_level": "low", "all_numbers_verified": False, "policy_check_passed": True},
    {"name": "draft that skipped the latest policy rules", "impact_level": "medium", "all_numbers_verified": True, "policy_check_passed": False},
]

for case in cases:
    print(f"{approve_finance_output(case):22s} <- {case['name']}")

send_to_manager        <- routine expense summary
human_review_required  <- credit limit change recommendation
reject                 <- draft memo with an unverified figure
reject                 <- draft that skipped the latest policy rules


Note that a **high-impact** task (like recommending a credit limit change) always routes to a human -- even though every number checked out and the policy check passed. The agent's job stopped at *preparing a well-supported recommendation*, not deciding.

An **audit trail** is the main control alongside approval -- every action logged, so any decision can be traced back to its inputs:

In [3]:
audit_log = []


def record_finance_action(task_id: str, actor: str, tools_called: list[str], data_read: list[str],
                           rules_applied: list[str], outcome: str, approved_by: str | None = None):
    audit_log.append({
        "task_id": task_id, "actor": actor, "tools_called": tools_called,
        "data_read": data_read, "rules_applied": rules_applied,
        "outcome": outcome, "approved_by": approved_by,
    })


record_finance_action(
    task_id="credit-note-882", actor="researcher_agent",
    tools_called=["exposure_api", "policy_lookup"], data_read=["account_exposure_2026Q1"],
    rules_applied=["credit_policy_v7#thresholds"], outcome="drafted_for_review",
)
record_finance_action(
    task_id="credit-note-882", actor="human_credit_officer",
    tools_called=[], data_read=[], rules_applied=[],
    outcome="approved", approved_by="j.martinez",
)

for entry in audit_log:
    print(entry)

{'task_id': 'credit-note-882', 'actor': 'researcher_agent', 'tools_called': ['exposure_api', 'policy_lookup'], 'data_read': ['account_exposure_2026Q1'], 'rules_applied': ['credit_policy_v7#thresholds'], 'outcome': 'drafted_for_review', 'approved_by': None}
{'task_id': 'credit-note-882', 'actor': 'human_credit_officer', 'tools_called': [], 'data_read': [], 'rules_applied': [], 'outcome': 'approved', 'approved_by': 'j.martinez'}
